# 01 — Load Amazon Reviews 2023 (Video Games)

Raw JSONL in `data/raw/` → Parquet in `data/interim/` + a 10% user sample for development.

## Step 2 — Load reviews in chunks

In [1]:
import sys

import pandas as pd

sys.path.insert(0, "../src")
from shoprank.config import load_config
from shoprank.data.load import load_items, load_reviews, sample_users

cfg = load_config("../config.yaml")
RAW, INTERIM = cfg.path("raw"), cfg.path("interim")
reviews = load_reviews(RAW / f"{cfg['category']}.jsonl.gz")

In [2]:
print(reviews.shape)
print("users:", reviews["user_id"].nunique(), " items:", reviews["parent_asin"].nunique())
print(reviews["timestamp"].min(), "->", reviews["timestamp"].max())
reviews.head()

(4624615, 8)


users: 2766656  items: 137249
1998-11-17 19:58:49 -> 2023-09-12 09:02:35.529000


,user_id,parent_asin,rating,title,text,timestamp,verified_purchase,helpful_vote
0,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B07DK1H3H5,4,It’s pretty sexual. Not my fav,I’m playing on ps5 and it’s interesting. It’s...,2020-12-17 06:33:24.795,True,0
1,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B07SRWRH5D,5,Good. A bit slow,Nostalgic fun. A bit slow. I hope they don’t...,2020-04-16 15:31:54.941,False,1
2,AGXVBIUFLFGMVLATYXHJYL4A5Q7Q,B07MFMFW34,5,... an order for my kids & they have really en...,This was an order for my kids & they have real...,2017-03-30 12:37:11.000,True,0
3,AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q,B0BCHWZX95,5,Great alt to pro controller,"These work great, They use batteries which is ...",2019-12-29 16:40:34.017,True,0
4,AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q,B00HUWA45W,5,solid product,I would recommend to anyone looking to add jus...,2015-03-29 01:18:52.000,True,0


## Step 3 — Load metadata

Keep only the columns we use; `images`, `videos`, `bought_together` are dropped. `price` arrives as a mix of floats, `None` and the odd string, so it is coerced to numeric.

In [3]:
items = load_items(RAW / f"meta_{cfg['category']}.jsonl.gz")

print(items.shape, " unique parent_asin:", items["parent_asin"].nunique())
print("review items with metadata:", reviews["parent_asin"].isin(items["parent_asin"]).mean().round(4))
items.head()

(137269, 9)  unique parent_asin: 137269


review items with metadata: 1.0


,parent_asin,title,features,description,price,store,categories,average_rating,rating_number
0,B000FH0MHO,Dash 8-300 Professional Add-On,[Features Dash 8-300 and 8-Q300 ('Q' rollout l...,[The Dash 8-300 Professional Add-On lets you p...,NaN,Aerosoft,"[Video Games, PC, Games]",5.0,1
1,B00069EVOG,Phantasmagoria: A Puzzle of Flesh,[Windows 95],[],NaN,Sierra,"[Video Games, PC, Games]",4.1,18
2,B00Z9TLVK0,NBA 2K17 - Early Tip Off Edition - PlayStation 4,[The #1 rated NBA video game simulation series...,[Following the record-breaking launch of NBA 2...,58.00,2K,"[Video Games, PlayStation 4, Games]",4.3,223
3,B07SZJZV88,Nintendo Selects: The Legend of Zelda Ocarina ...,[Authentic Nintendo Selects: The Legend of Zel...,[],37.42,Amazon Renewed,"[Video Games, Legacy Systems, Nintendo Systems...",4.9,22
4,B002WH4ZJG,Thrustmaster Elite Fitness Pack for Nintendo Wii,"[Includes (9) Total Accessories, Pedometer, Wi...",[The Thrustmaster Motion Plus Elite Fitness Pa...,NaN,THRUSTMASTER,"[Video Games, Legacy Systems, Nintendo Systems...",3.0,3


## Step 4 — Save as Parquet

In [4]:
interim = INTERIM
interim.mkdir(parents=True, exist_ok=True)
reviews.to_parquet(interim / "reviews.parquet", index=False)
items.to_parquet(interim / "items.parquet", index=False)

for f in sorted(interim.glob("*.parquet")):
    print(f.name, f"{f.stat().st_size / 1e6:.0f} MB")

items.parquet 93 MB
items_sample.parquet 48 MB
reviews.parquet 1069 MB
reviews_sample.parquet 105 MB


## Step 5 — 10% development sample (by user, not by row)

Sampling users keeps each sampled user's full review history intact.

In [5]:
reviews_sample = sample_users(reviews, frac=0.10, random_state=cfg.seed)
items_sample = items[items["parent_asin"].isin(reviews_sample["parent_asin"])]

reviews_sample.to_parquet(interim / "reviews_sample.parquet", index=False)
items_sample.to_parquet(interim / "items_sample.parquet", index=False)

n_users, n_sample = reviews["user_id"].nunique(), reviews_sample["user_id"].nunique()
print("sample users:", n_sample, f"({n_sample / n_users:.1%})")
print("sample reviews:", len(reviews_sample), f"({len(reviews_sample) / len(reviews):.1%})")
print("sample items:", len(items_sample))

sample users: 276666 (10.0%)
sample reviews: 460834 (10.0%)
sample items: 60762


## Sanity check — reload from Parquet

In [6]:
check = pd.read_parquet(interim / "reviews_sample.parquet")
print(check.dtypes)
assert check.groupby("user_id").size().sum() == len(reviews[reviews["user_id"].isin(check["user_id"].unique())])

user_id                         str
parent_asin                     str
rating                        int64
title                           str
text                            str
timestamp            datetime64[ms]
verified_purchase              bool
helpful_vote                  int64
dtype: object
